## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [2]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
nasdaq100_ca = pd.read_csv('nasdaq100_CA.csv', index_col=0)
# Use as many cells as you like
print(nasdaq100_ca.head())
nasdaq100_price_change = pd.read_csv('nasdaq100_price_change.csv', index_col=0)
print(nasdaq100_price_change.head())
nasdaq100_ca['ytd']=nasdaq100_price_change['ytd']
nasdaq100_ca = nasdaq100_ca.reset_index()
print(nasdaq100_ca.head())



prompt ='Classify the stock company based on their sector. Just return their sector name, do not include any explaination. The following values you can classify with is Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, or Financial.'
nasdaq = nasdaq100_ca.copy()
sectors = []

for company in nasdaq['name']:
    response=client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role':'system', 'content':f'{prompt} Classify this company {company}' 
        }]
      
    )
    sectors.append(response.choices[0].message.content.strip())
print(response.choices[0].message.content)
nasdaq100_ca['sector'] = sectors
nasdaq = nasdaq100_ca.copy()



text = """
Provide summary information about the selected Nasdaq companies.
Recommend the TWO best-performing sectors based on YTD performance.
Recommend at least TWO companies from each selected sector.
Explain your recommendations using the YTD values.
"""
stock_data = nasdaq[['name', 'sector', 'ytd']].to_string(index=False)
response= client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'system', 'content':f'{text} Recommend three best sectors and few best companies per sector{stock_data}'
               }]
)
stock_recommendations = response.choices[0].message.content
print(stock_recommendations)
    


                     name        headQuarter  ...      cik     founded
symbol                                        ...                     
AAPL           Apple Inc.      Cupertino, CA  ...   320193  1976-04-01
ABNB               Airbnb  San Francisco, CA  ...  1559720  2008-08-01
ADBE           Adobe Inc.       San Jose, CA  ...   796343  1982-12-01
ADSK             Autodesk     San Rafael, CA  ...   769397  1982-01-30
AMAT    Applied Materials    Santa Clara, CA  ...     6951  1967-11-10

[5 rows x 5 columns]
            1D       5D        1M  ...         5Y         10Y           max
symbol                             ...                                     
AAPL   -1.7254 -8.30086  -6.20411  ...  245.42031   976.99441  139245.53954
ABNB    2.1617 -2.21919   9.88336  ...   -1.04347    -1.04347      -1.04347
ADBE    0.5409 -1.77817   9.16191  ...  109.05718  1024.69214  251030.66399
ADI     0.9291 -4.03352   2.58486  ...   92.81874   286.77518   26012.63736
ADP     2.0589  2.35462  